# Simulation Sanity Check

Load a saved simulation pickle, validate its numerical and process-level outputs, and inspect standard figures.

## Execution paths

1. **Inspection (default):** load the selected pickle and run every following section without changing it.
2. **Optional re-run:** set `RERUN_SIMULATION = True`, define any parameter overrides in the dedicated section, and re-run the loaded configuration before validation.

Run the notebook from the campaign directory so pickle discovery starts at the correct location. The selected pickle should contain detailed trajectories; performance-mode results can lack data required by some plots.

**Workflow:**
1. Setup and configuration
2. Load or optionally re-run the simulation
3. KPI, feasibility, and elemental-closure diagnostics
4. Batch, kinetic, and process-level visualizations
5. Interpretation checklist

## 1. Setup and Configuration

In [3]:
from pathlib import Path

from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.simulation_data import SimulationData

print("✓ Imports successful")

✓ Imports successful


### 1.1 Discover Available Simulation Pickles

Scan the campaign directory recursively for saved simulation pickles. Configure the selected result and inspection scope in the next cell.

In [4]:
# ── Scan this campaign directory recursively for simulation pickles ───────────
CAMPAIGN_DIR = Path.cwd()
PICKLE_FILES = sorted(
    path for path in CAMPAIGN_DIR.rglob("*.pkl")
    if path.is_file()
)

if not PICKLE_FILES:
    raise FileNotFoundError(
        f"No .pkl files found below the campaign directory: {CAMPAIGN_DIR.resolve()}"
    )

print(f"Found {len(PICKLE_FILES)} simulation pickle(s) below {CAMPAIGN_DIR.resolve()}:\n")
for number, path in enumerate(PICKLE_FILES, start=1):
    print(f"[{number:>3}] {path.relative_to(CAMPAIGN_DIR)}")


Found 60 simulation pickle(s) below C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_3_dynamic_stoichiometry_GFI:

[  1] 01_FedBatch\export\fedbatch_constant_opt_20260731_0017\fedbatch_constant_opt_20260731_0017.pkl
[  2] 01_FedBatch\export\fedbatch_constant_p2_20260815_1127\opt_20260815_1127.pkl
[  3] 01_FedBatch\export\fedbatch_constant_p2_20260819_1134\opt_20260819_1134.pkl
[  4] 01_FedBatch\export\fedbatch_constant_s2_20260808_1151\opt_20260808_1151.pkl
[  5] 01_FedBatch\export\fedbatch_csfr_opt_20260731_0446\fedbatch_csfr_opt_20260731_0446.pkl
[  6] 01_FedBatch\export\fedbatch_csfr_p2_20260815_1527\opt_20260815_1527.pkl
[  7] 01_FedBatch\export\fedbatch_csfr_p2_20260819_1554\opt_20260819_1554.pkl
[  8] 01_FedBatch\export\fedbatch_csfr_s2_20260808_1332\opt_20260808_1332.pkl
[  9] 01_FedBatch\export\fedbatch_exponential_opt_20260731_0945\fedbatch_exponential_opt_20260731_0945.pkl
[ 10] 01_FedBatch\export\fedbatch_exponential_p2_20260815_1858\o

In [5]:
# ── Select the saved simulation and inspection scope ──────────────────────────
# Set this to one of the numbers printed by the preceding discovery cell.
PICKLE_NUMBER = 23

# Individual batch used by the batch and kinetic figure sections.
BATCH_NUMBER = 1

# None includes every batch in process-level overlays; use an integer to limit them.
MAX_BATCHES = None

# False preserves the loaded result. Set True only to apply overrides and re-run.
RERUN_SIMULATION = False

if not isinstance(PICKLE_NUMBER, int) or isinstance(PICKLE_NUMBER, bool):
    raise TypeError("PICKLE_NUMBER must be an integer.")
if not 1 <= PICKLE_NUMBER <= len(PICKLE_FILES):
    raise ValueError(
        f"PICKLE_NUMBER must be between 1 and {len(PICKLE_FILES)}; "
        f"received {PICKLE_NUMBER}."
    )
if not isinstance(BATCH_NUMBER, int) or isinstance(BATCH_NUMBER, bool) or BATCH_NUMBER < 1:
    raise ValueError("BATCH_NUMBER must be a positive integer.")
if MAX_BATCHES is not None and (
    not isinstance(MAX_BATCHES, int) or isinstance(MAX_BATCHES, bool) or MAX_BATCHES < 1
):
    raise ValueError("MAX_BATCHES must be None or a positive integer.")

PICKLE_PATH = PICKLE_FILES[PICKLE_NUMBER - 1]

print(f"Selected pickle [{PICKLE_NUMBER}]: {PICKLE_PATH.relative_to(CAMPAIGN_DIR)}")
print(f"Batch for detailed plots: {BATCH_NUMBER}")
print(f"Optional re-run enabled: {RERUN_SIMULATION}")

Selected pickle [23]: 02_Batch\export\batch_p2_20260820_1127\opt_20260820_1127.pkl
Batch for detailed plots: 1
Optional re-run enabled: False


## 2. Load the Selected Simulation

This is the default inspection path. The optional re-run section below replaces `sim` and `data` only when enabled.

In [6]:
sim = CellCultureSimulation.load_pickle(PICKLE_PATH, base_export_dir=PICKLE_PATH.parent)
data = sim.data

if not 1 <= BATCH_NUMBER <= len(data.batches):
    raise ValueError(
        f"BATCH_NUMBER must be between 1 and {len(data.batches)} for the selected simulation; "
        f"received {BATCH_NUMBER}."
    )

print(f"✓ Loaded: {data.simulation_type}")
print(f"  Source           : {PICKLE_PATH.resolve()}")
print(f"  Batches          : {len(data.batches)}")
print(f"  Simulation time  : {data.simulated_window_totals_stats.get('total_simulation_time_h', 'N/A'):.1f} h")
print(f"  Parameters name  : {data.parameters.metadata.get('name', 'N/A') if data.parameters else 'N/A'}")

CellCultureSimulation loaded from: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_3_dynamic_stoichiometry_GFI\02_Batch\export\batch_p2_20260820_1127\opt_20260820_1127.pkl
✓ Loaded: Batch (startup Fedbatch)
  Source           : C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_3_dynamic_stoichiometry_GFI\02_Batch\export\batch_p2_20260820_1127\opt_20260820_1127.pkl
  Batches          : 8
  Simulation time  : 960.0 h
  Parameters name  : base_parameters


### 2.1 Optional Parameter Overrides and Re-run

Leave `RERUN_SIMULATION = False` to inspect the loaded result unchanged. To explore a changed configuration, enable it in the configuration cell and add parameter-path/value pairs to `PARAMETER_OVERRIDES` in the next cell. The re-run replaces `sim` and `data` for every subsequent diagnostic and figure.

### 2.2 Run Optional Overrides

In [7]:
# Keep this mapping empty to re-run the saved configuration without changes.
# Uncomment or add entries only when RERUN_SIMULATION is True.
PARAMETER_OVERRIDES = {
    # "fed_batch.C_Gln_feed": 10,
    # "fed_batch.C_AA_feed": 10,
    # "initial_conditions.C_Glc_init": 40,
    # "initial_conditions.C_Gln_init": 20,
    # "fed_batch.CSFR": 5e-12,
    # "perfusion.CSPR": 10.0e-11,
}

if RERUN_SIMULATION:
    for parameter_path, value in PARAMETER_OVERRIDES.items():
        sim.parameters[parameter_path] = value

    sim = CellCultureSimulation(
        parameters=sim.parameters,
        kinetics=sim.kinetics,
        gas_controller=sim.gas_controller,
        operating_mode=sim.operating_mode,
        feed_strategy=sim.feed_strategy,
        perfusion_strategy=sim.perfusion_strategy,
        stoichiometry_mode=sim.stoichiometry_mode,
        performance_mode=sim.performance_mode,
        solver_method="LSODA",
    )
    data = sim.run_simulation()
    print(f"✓ Re-ran simulation with {len(PARAMETER_OVERRIDES)} override(s).")
else:
    print("✓ Inspection mode: using the loaded simulation unchanged.")

✓ Inspection mode: using the loaded simulation unchanged.


## 3. Validation and Diagnostics

Review optimization feasibility, performance, and elemental closure before interpreting figures.

### 3.1 Optimization Feasibility Check

This uses the same `ObjectiveFunction` path and rejection criteria as differential evolution. A pass means the result is feasible for optimization; it does not replace scientific review.

In [8]:
# Evaluate the current simulation with the same ObjectiveFunction implementation
# used by differential evolution. This avoids maintaining a second, drifting copy
# of the optimization rejection logic in the notebook.

from animakinetica.analysis.optimization import ObjectiveFunction
import numpy as np

WALL_TIMEOUT_S = 10.0  # Match the campaign optimization timeout when diagnosing runtime rejections.

# ObjectiveFunction needs objectives and weights even though this cell only
# evaluates feasibility. Use a harmless available numeric KPI with zero weight.
validation_function = ObjectiveFunction(
    baseline_simulation=sim,
    opt_config={},
    objectives=["STY_kg_m3_day"],
    weights=[0.0],
    timeout=WALL_TIMEOUT_S,
    baseline_values=None,
    debug_rejections=True,
)

# An empty optimization configuration maps this empty vector to the current
# baseline parameters, then runs the same temporary-simulation path and every
# ObjectiveFunction rejection check. Valid evaluations return a non-positive
# score; soft constraint penalties and hard rejections return positive scores.
filter_score = float(validation_function(np.array([], dtype=float)))
passes_filter = filter_score <= 0.0

print("ObjectiveFunction filter check:")
print(f"  {'PASS' if passes_filter else 'FAIL'}  valid ObjectiveFunction score <= 0")
print(f"  Score: {filter_score:.6g}")

# Supplementary diagnostics only; all pass/fail decisions remain in
# ObjectiveFunction. The fields below expose inputs to its current filters.
simulated_totals = data.simulated_window_totals_stats
expected_window_h = validation_function._get_expected_simulation_window_target_h()
soft_violations = validation_function._get_soft_constraint_violations(
    data,
    simulated_totals,
)

try:
    closure = data.compute_boundary_closure()
    elem_res = {
        element: abs(values["relative_residual_pct"])
        for element, values in closure["elements"].items()
    }
except (ValueError, KeyError):
    elem_res = None

total_base_in_L = float(simulated_totals.get("total_base_in_L", 0.0) or 0.0)
total_acid_in_L = float(simulated_totals.get("total_acid_in_L", 0.0) or 0.0)
total_medium_in_L = float(simulated_totals.get("total_medium_in_L", 0.0) or 0.0)
acid_base_fraction = (
    (total_base_in_L + total_acid_in_L) / total_medium_in_L
    if total_medium_in_L > 0.0 else None
)

print("\nDiagnostics:")
print(f"  total_simulation_time_h = {float(simulated_totals.get('total_simulation_time_h', 0.0) or 0.0):.3f}")
print(f"  expected_simulation_window_h = {expected_window_h:.3f}")
print(f"  total_active_cultivation_time_h = {float(simulated_totals.get('total_active_cultivation_time_h', 0.0) or 0.0):.3f}")
print(f"  projection_window_h = {simulated_totals.get('projection_window_h')}")
print(f"  projection_target_time_h = {simulated_totals.get('projection_target_time_h')}")
print(f"  number_of_batches = {simulated_totals.get('number_of_batches')}")
print(f"  total_DCM_kg = {simulated_totals.get('total_DCM_kg', data.summary_stats.get('total_DCM_kg'))}")
print(f"  last_batch_termination_reason = {simulated_totals.get('last_batch_termination_reason')}")
print(f"  unique_batch_termination_reasons = {simulated_totals.get('unique_batch_termination_reasons')}")
print(f"  batch_final_volume_min_fraction = {simulated_totals.get('batch_final_volume_min_fraction')}")
print(f"  batch_final_volume_max_fraction = {simulated_totals.get('batch_final_volume_max_fraction')}")
print(f"  last_batch_final_volume_fraction = {simulated_totals.get('last_batch_final_volume_fraction')}")
print(f"  total_base_in_L = {total_base_in_L:.6g}")
print(f"  total_acid_in_L = {total_acid_in_L:.6g}")
print(f"  total_medium_in_L = {total_medium_in_L:.6g}")
print(f"  acid_base_fraction = {acid_base_fraction if acid_base_fraction is not None else 'N/A'}")
print(f"  phase_active_runtime_h = {simulated_totals.get('phase_active_runtime_h')}")
print(f"  batch_phase_active_runtime_h = {simulated_totals.get('batch_phase_active_runtime_h')}")
print(f"  ObjectiveFunction soft_constraint_violations = {soft_violations}")
if elem_res is not None:
    print(
        "  boundary closure |relative_residual_pct| "
        f"(soft limit {ObjectiveFunction._MAX_CLOSURE_PCT:.0f}%) = {elem_res}"
    )

run_simulation execution time: 585.241 ms
ObjectiveFunction filter check:
  PASS  valid ObjectiveFunction score <= 0
  Score: -0

Diagnostics:
  total_simulation_time_h = 960.000
  expected_simulation_window_h = 960.000
  total_active_cultivation_time_h = 0.000
  projection_window_h = 960.0
  projection_target_time_h = 7680.0
  number_of_batches = 8
  total_DCM_kg = None
  last_batch_termination_reason = Maximum time reached
  unique_batch_termination_reasons = ['Aa depletion', 'Ammonia ic100', 'Glucose depletion', 'Maximum time reached']
  batch_final_volume_min_fraction = 1.0
  batch_final_volume_max_fraction = 1.00168216055737
  last_batch_final_volume_fraction = 1.0
  total_base_in_L = 5.69661
  total_acid_in_L = 0
  total_medium_in_L = 15360.8
  acid_base_fraction = 0.0003708527872027257
  phase_active_runtime_h = {'FEDBATCH': 23.33917544875044, 'BATCH': 936.6608245512496}
  batch_phase_active_runtime_h = [{'batch_number': 1, 'BATCH': 145.2511437804494, 'FEDBATCH': 23.339175448750

### 3.2 Performance Summary

Inspect the primary KPIs and cultivation-level totals.

In [9]:
print("Performance Metrics:")
for key, value in data.summary_stats.items():
    if isinstance(value, float):
        print(f"  {key:<45} {value:.4f}")
    else:
        print(f"  {key:<45} {value}")

Performance Metrics:
  total_DCM_harvested_kg                        242.4713
  total_WCM_harvested_kg                        1616.4753
  STY_kg_m3_day                                 0.3789
  STY_WCM_kg_m3_day                             2.5257
  medium_conversion_efficiency_m3_kg            0.5068
  medium_conversion_efficiency_m3_kg_WCM        0.0760
  protein_conversion_efficiency_kg_kg           2.6351
  feed_conversion_efficiency_kg_kg              5.1849
  feed_conversion_efficiency_kg_kg_WCM          0.7777


In [10]:
# Cultivation summary table
data.get_cultivation_summary_df()

,Batch No.,Termination Reason,Cultivation Time (h),Batch Start Time (h),Batch End Time (h),Accum. Cultivation Time (d),Final Glucose (mmol/L),Final Glutamine (mmol/L),Final Lactate (mmol/L),Final Ammonia (mmol/L),...,Accum. Acid Addition Use (L),Accum. Total Inlet Use (L),Harvest Volume (L),Accum. Harvest Volume (L),Final Cell Density (10^6 cells/mL),Final Cell Density (g/L),Cells Harvested (10^12 Cells),Cells Harvested (kg),Accum. Cell Harvest (10^12 Cells),Accum. Biomass Harvest (kg)
0,1,Ammonia ic100,168.590319,0.000000,168.590319,7.024597,28.468914,2.022459,24.722610,8.150000,...,0.0,2003.364316,1908.625961,1908.625961,12.666464,1.899970,24.175541,3.626331,24.175541,3.626331
1,2,Glucose depletion,116.414035,168.590319,285.004355,11.875181,0.001000,7.978692,24.292120,8.008086,...,0.0,3910.961557,1913.182460,3821.808421,13.459880,2.018982,25.751206,3.862681,49.926748,7.489012
2,3,Aa depletion,113.136133,285.004355,398.140487,16.589187,0.909740,8.666487,22.555156,7.435482,...,0.0,5821.807717,1909.119150,5730.927571,13.204102,1.980615,25.208204,3.781231,75.134952,11.270243
3,4,Aa depletion,112.453318,398.140487,510.593805,21.274742,1.285402,8.748327,22.207315,7.320814,...,0.0,7730.926867,1908.909524,7639.837095,13.173715,1.976057,25.147431,3.772115,100.282382,15.042357
4,5,Aa depletion,112.363478,510.593805,622.957284,25.956553,1.343812,8.757714,22.157119,7.304266,...,0.0,9639.836390,1908.899976,9548.737071,13.172335,1.975850,25.144669,3.7717,125.427051,18.814058
5,6,Aa depletion,112.350720,622.957284,735.308003,30.637833,1.352554,8.758976,22.149774,7.301845,...,0.0,11548.736366,1908.899541,11457.636612,13.172272,1.975841,25.144543,3.771682,150.571595,22.585739
6,7,Aa depletion,112.348857,735.308003,847.656861,35.319036,1.353851,8.759156,22.148692,7.301488,...,0.0,13457.635908,1908.899521,13366.536133,13.172269,1.975840,25.144538,3.771681,175.716133,26.35742
7,8,Maximum time reached,112.343139,847.656861,960.000000,40.000000,1.357764,8.759867,22.145714,7.300506,...,0.0,15366.535429,2000.000000,15366.536133,13.171639,1.975746,26.343277,3.951492,202.059410,30.308912


In [11]:
process_stoich_closure = data.compute_stoichiometric_closure()
process_stoich_closure_df = data.build_closure_summary_df(process_stoich_closure)

process_stoich_biomass = next(
    spec for spec in process_stoich_closure['component_specs']
    if spec['label'] == 'Dry biomass'
)

print('Integrated stoichiometric CHON closure:')
display(process_stoich_closure_df)
print(f"Max absolute elemental residual: {process_stoich_closure['max_abs_residual']:.6g} mol-atoms")

Integrated stoichiometric CHON closure:


,Element,Reactants (mol-atoms),Products (mol-atoms),Residual (mol-atoms),Relative Residual (%)
0,C,4294.498162,4294.448082,-0.050080,-0.001166
1,H,8641.950468,8641.833335,-0.117133,-0.001355
2,O,7177.449273,7177.416141,-0.033132,-0.000462
3,N,412.758709,412.741753,-0.016956,-0.004108


Max absolute elemental residual: 0.117133 mol-atoms


In [12]:
process_external_elemental_closure = data.compute_boundary_closure()
process_external_elemental_closure_df = data.build_closure_summary_df(process_external_elemental_closure)

process_dry_biomass = next(
    spec for spec in process_external_elemental_closure['component_specs']
    if spec['label'] == 'DCM out'
 )

print('Whole-process external-boundary CHON closure:')
display(process_external_elemental_closure_df)
print(f"Max absolute elemental residual: {process_external_elemental_closure['max_abs_residual']:.6g} mol-atoms")

Whole-process external-boundary CHON closure:


,Element,Reactants (mol-atoms),Products (mol-atoms),Residual (mol-atoms),Relative Residual (%)
0,C,5.832206e+03,5.832182e+03,-0.024253,-0.000416
1,H,1.716778e+06,1.716778e+06,-0.080474,-0.000005
2,O,8.615746e+05,8.618401e+05,265.429597,0.030807
3,N,6.776778e+02,6.776607e+02,-0.017095,-0.002523


Max absolute elemental residual: 265.43 mol-atoms


## 4. Batch-State and Operating-Control Figures

Use `BATCH_NUMBER` from the configuration section to inspect biomass, metabolites, volume/feed, gas transfer, and pH control for one batch.

## 5. Batch Plots

In [13]:
# Cell density & metabolites
fig = data.plot_biomass_metabolites(batch_number=BATCH_NUMBER)
fig.show()

In [14]:
# Volume & feed / dilution rate
fig = data.plot_volume_feed(batch_number=BATCH_NUMBER)
fig.show()

In [15]:
# Gas transfer (DO, agitation, gas flow, O2 fraction)
fig = data.plot_gas_transfer(batch_number=BATCH_NUMBER)
fig.show()

In [16]:
# pH plots
fig = data.plot_pH_control(batch_number=BATCH_NUMBER)
fig.show()

## 5. Kinetic-Behavior Figures

Review nutrient limitation, inhibition, kinetic rates, and stoichiometric yields for the selected batch.

In [17]:
# Nutrient limitation factors (Monod terms)
fig = data.plot_nutrient_factors(batch_number=BATCH_NUMBER)
fig.show()

In [18]:
# Inhibition factors (lactate, ammonia, O2, osmolality)
fig = data.plot_inhibition_factors(batch_number=BATCH_NUMBER)
fig.show()

In [19]:
# Specific rates (mu, q_Glc, q_Gln, q_Lac, q_O2, etc.)
fig = data.plot_kinetic_rates(batch_number=BATCH_NUMBER)
fig.show()

In [20]:
# Stoichiometric yield coefficients (Y_X/Glc, Y_Lac/Glc, Y_O2/Glc, etc.)
fig = data.plot_stoichiometric_yields(batch_number=BATCH_NUMBER)
fig.show()

## 6. Multi-Batch Process Figures

Use `MAX_BATCHES` from the configuration section to constrain batch overlays and cumulative totals.

In [21]:
# Overlaid state variable profiles across batches
figs = data.plot_profiles(max_batches=MAX_BATCHES)
for fig in figs:
    fig.show()

In [22]:
# Cumulative harvest (biomass & volume) across batches
fig = data.plot_cumulative_harvest(max_batches=MAX_BATCHES)
fig.show()

In [23]:
# Cumulative medium use
fig = data.plot_cumulative_medium_use(max_batches=MAX_BATCHES)
fig.show()

In [24]:
# Cumulative gas use
fig = data.plot_cumulative_gas_use(max_batches=MAX_BATCHES)
fig.show()

## 7. Interpretation Checklist

Before accepting a result:

- Confirm the selected pickle and whether the optional re-run path was enabled.
- Investigate a failed optimization feasibility check, especially its rejection diagnostics and termination reason.
- Review elemental residuals in both closure reports; large residuals require mass-balance investigation.
- Check the selected-batch figures for unexpected depletion, volume limits, gas-control saturation, or pH-control behavior.
- If a plot lacks data, verify that the pickle was generated with `performance_mode=False` and contains detailed trajectories.